<a href="https://colab.research.google.com/github/Jyoti1706/AI-Agent/blob/main/Context_Engineering_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install sentence-transformers faiss-cpu langchain pypdf langchain_community langchain-text-splitters


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
  print(filename)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
pdf_file = list(uploaded.keys())[0]
loader = PyPDFLoader(pdf_file)
documents = loader.load()
print(f"number of pages : {len(documents)}")

In [ ]:
print(documents[1])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)
print(f"total chunk created: {len(chunks)}")

In [ ]:
print(chunks[0].page_content)

In [ ]:
# embeddings from chunks
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")  ## 22 Milion parameter
embeddings = model.encode([chunk.page_content for chunk in chunks], convert_to_tensor=True)
dimension = embeddings.shape[1]
print(f"embeddings dimension: {dimension}")
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [ ]:
import matplotlib.pyplot as plt
def visualize_embeddings(vector, chunk_text=None, max_text_chars=500):
  vector = np.array(vector)
  normalized = (vector-vector.min())


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# Convert embeddings to a NumPy array if it's a PyTorch tensor
embeddings_np = embeddings.cpu().numpy() if hasattr(embeddings, 'cpu') else np.array(embeddings)

# Apply PCA to reduce dimensions to 2 for visualization
pca = PCA(n_components=2)
reduced_embeddings = pca.fit_transform(embeddings_np)

# Plot the reduced embeddings
plt.figure(figsize=(10, 8))
plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], s=10)
plt.title('2D PCA Visualization of Document Chunks')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.show()